In [3]:
%cd /workspace
#/EBES

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from pathlib import Path
import optuna 
from ebes.pipeline.utils import optuna_df
from optuna.trial import TrialState

/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/workspace


/usr/local/lib/python3.10/dist-packages/torch/cuda/__init__.py:619: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [11]:
def get_run(number, specify="best", rewrite=False):
    path = Path(f"log/{dataset}/{method}/optuna/{number}")
    print(pd.read_csv(path / "results.csv"))
    print((path / "params.txt").read_text())
    save_path = Path(f"configs/specify/{dataset}/{method}")
    save_path.mkdir(parents=True, exist_ok=True)
    save_path = (save_path / f"{specify}.yaml")
    if not rewrite:
        assert not save_path.exists()
    save_path.write_text((path / "params.txt").read_text())

def prepare_data(dataset, method):
    path = Path(f"log/{dataset}/{method}/optuna")
    df, study = optuna_df(path)
    value_pack = ["values_0", "values_1", "values_2"]#, "values_3"]
    col_to_drop = ["datetime_start", "datetime_complete", "system_attrs_fixed_params", "state", *value_pack]
    col_params = [*value_pack, "duration"] + [col for col in df if "params_" in col]
    col_user = [*value_pack, "duration"] + [col for col in df if "user" in col]
    df["duration"] = df["duration"].dt.total_seconds()
    return df, study, col_user, col_params

In [14]:
dataset = "twitter"
method = "SimCLR"
df, study, col_user, col_params = prepare_data(dataset, method)

print(df.columns)
core_param = "values_2"
print(df.shape, df[~df[core_param].isna()].shape)
test_cols = [col for col in col_user if ("test" in col)]
df[~df[core_param].isna()].sort_values(core_param, ascending=False).iloc[:10, ][[core_param] + test_cols]

Index(['values_0', 'values_1', 'values_2', 'datetime_start',
       'datetime_complete', 'duration',
       'params_data.loaders.unsupervised_train.batch_size',
       'params_model.aggregation.name',
       'params_model.encoder.params.hidden_size',
       'params_model.encoder.params.num_layers',
       'params_model.preprocess.params.cat_emb_dim',
       'params_model.preprocess.params.num_emb_dim',
       'params_model.preprocess.params.num_norm',
       'params_model.preprocess.params.time_process',
       'params_optimizer.params.lr', 'params_optimizer.params.weight_decay',
       'params_unsupervised_loss.params.temperature', 'user_attrs_loss_mean',
       'user_attrs_loss_std', 'user_attrs_memory_after_mean',
       'user_attrs_memory_after_std',
       'user_attrs_target__anomaly__global__roc_auc__logreg_torch__roc_auc_mean',
       'user_attrs_target__anomaly__global__roc_auc__logreg_torch__roc_auc_std',
       'user_attrs_target__anomaly__global__roc_auc_mean',
       'user_

/workspace/ebes/pipeline/utils.py:168: ExperimentalWarning: JournalStorage is experimental (supported from v3.1.0). The interface can change in the future.
  storage = JournalStorage(JournalFileStorage(f"{path}/study.log"))


,values_2
20,0.996089
53,0.995344
44,0.995263
22,0.995148
49,0.995104
43,0.994511
42,0.994471
35,0.993629
23,0.993538
48,0.993536


In [15]:
#get_run(33, specify="best_Specific_name")

In [16]:
failed = df[(df["state"] != "COMPLETE") | (df[col_user].isna().any(axis=1))][col_user].index
print(failed)
for fail in df[(df["state"] != "COMPLETE") | (df[col_user].isna().any(axis=1))][col_user].index:
    error_path = Path(f"/home/dev/24/es-bench/log/{dataset}/{method}/optuna/{fail}/ERROR.txt")
    if error_path.exists():
        error = error_path.read_text()
        print(fail, error.split("\n")[-2])
    else:
        print(df.loc[fail])

Index([], dtype='int64')


In [17]:
optuna.visualization.plot_optimization_history(study, target = lambda t: t.values[int(core_param[-1])])

/usr/local/lib/python3.10/dist-packages/optuna/visualization/_utils.py:67: UserWarning: `target` is specified, but `target_name` is the default value, 'Objective Value'.
  warnings.warn(


### Params influence

In [12]:
target_objective_index = int(core_param[-1])
trials = study.trials
trials = [trial for trial in trials if trial.state == TrialState.COMPLETE]
plotted_trials = sorted(trials, key=lambda t: t.values[target_objective_index])[:]
plotted_study = optuna.create_study()
for trial in plotted_trials:
    # Создаем "чистую" копию триала только с одним нужным значением
    single_value_trial = optuna.trial.create_trial(
        params=trial.params,
        distributions=trial.distributions,
        value=trial.values[target_objective_index], # Берем только одно значение
    )
    plotted_study.add_trial(single_value_trial)

[I 2026-03-25 14:25:59,572] A new study created in memory with name: no-name-8a006582-992e-40b1-955a-058b3457b930


In [13]:
target = None #lambda t: (t.user_attrs["memory_after_mean"])
target_name = "value"
fig = optuna.visualization.plot_param_importances(plotted_study, target=target, target_name=target_name)
print(fig._data[0]["x"][::-1])
print(fig._data[0]["y"][::-1])
take = 7
params = fig._data[0]["y"][-take:]
not_imp = list(set([col.replace("params_", "") for col in col_params]) - set(params) - {"duration", "value", "system_attrs_fixed_params"})
fig

[0.3891153008696473, 0.23428092270105336, 0.12152545883704131, 0.065409499970096, 0.05652143044810723, 0.04266429037168388, 0.025528082787406307, 0.023547212803750224, 0.021003449847572467, 0.011495160437361599, 0.008909190926280204]
['model.preprocess.params.cat_emb_dim', 'optimizer.params.weight_decay', 'model.preprocess.params.num_emb_dim', 'optimizer.params.lr', 'model.encoder.params.hidden_size', 'model.aggregation.name', 'model.encoder.params.num_layers', 'model.preprocess.params.num_norm', 'data.loaders.unsupervised_train.batch_size', 'model.emb_head.params.out_features', 'model.preprocess.params.time_process']


In [27]:
params

['model.emb_head.params.out_features',
 'model.preprocess.params.num_emb_dim',
 'model.encoder.params.num_layers',
 'model.preprocess.params.cat_emb_dim',
 'model.encoder.params.hidden_size',
 'optimizer.params.weight_decay',
 'optimizer.params.lr']

In [15]:
fig = optuna.visualization.plot_slice(plotted_study, target=target, target_name=target_name)

In [16]:
fig